In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Build Gene Length File for SETIA

**Goal**: Produce `BMMC_gene_length.txt` — BMMC equivalent of yeast pipeline's `GRN_ssTFs_Sc_gene_length.txt`.

**Yeast format** (verified from `GRN_input_acquisition.py` line 836):
- Single line, tab-separated
- One length value per gene in `Column_order` (= all 88 genes from bmmc_gene_list.tsv)
- No trailing tab

**Length definition for human**:
- We use **canonical transcript length** from GENCODE v44 (Ensembl primary)
- Canonical = MANE Select if available, else longest CDS-containing transcript
- This is consistent with how RNA-seq length normalization is conventionally done in human
- Yeast pipeline uses ORF length (no introns since yeast genes are mostly intronless), but for human, transcript length (with introns excluded) is the standard

In [ ]:
import gzip
import urllib.request
import pandas as pd
from pathlib import Path
from collections import defaultdict

GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
OUTPUT_DIR  = Path(f"{HUMAN_BASE}/GTEx_v11/gene_length_output");  OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR   = Path(f"{HUMAN_BASE}/GTEx_v11/gene_length_cache");   CACHE_DIR.mkdir(exist_ok=True)

# GENCODE v44 (same version as ChIP prior used for TSS)
GENCODE_URL = 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz'
GTF_PATH    = CACHE_DIR / 'gencode.v44.annotation.gtf.gz'

# Load gene list — SAME ordering as expression matrix and other priors
gene_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
all_genes = gene_df['gene_symbol'].tolist()
print(f'Total genes (matrix dim): {len(all_genes)}')
print(f'First 5: {all_genes[:5]}')
print(f'Last 5:  {all_genes[-5:]}')

In [ ]:
# Download GENCODE GTF if not cached
if not GTF_PATH.exists():
    print(f'Downloading GENCODE v44 GTF (~50 MB)...')
    urllib.request.urlretrieve(GENCODE_URL, GTF_PATH)
print(f'GTF file: {GTF_PATH.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Parse GTF — build per-gene transcript info
# We need: gene_name → transcript_id → transcript_length, is_canonical, has_CDS
# Strategy: collect all 'transcript' entries with their tags, compute lengths from 'exon' entries

gene_targets = set(all_genes)
transcripts = {}      # transcript_id -> {gene_name, tags, has_cds, length}
exon_lengths = defaultdict(int)   # transcript_id -> total exonic length

def parse_attrs(attr_str):
    """Parse GTF attribute string into dict."""
    attrs = {}
    for kv in attr_str.strip().rstrip(';').split(';'):
        kv = kv.strip()
        if not kv:
            continue
        if ' ' not in kv:
            continue
        key, val = kv.split(' ', 1)
        val = val.strip().strip('"')
        if key in attrs:
            # multi-value field like 'tag' — collect into list
            if isinstance(attrs[key], list):
                attrs[key].append(val)
            else:
                attrs[key] = [attrs[key], val]
        else:
            attrs[key] = val
    return attrs

print('Parsing GTF (this takes ~1 min)...')
import time
t0 = time.time()
with gzip.open(GTF_PATH, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.rstrip('\n').split('\t')
        if len(parts) < 9:
            continue
        feature = parts[2]
        if feature not in ('transcript', 'exon', 'CDS'):
            continue
        attrs = parse_attrs(parts[8])
        gene_name = attrs.get('gene_name')
        if gene_name not in gene_targets:
            continue
        tid = attrs.get('transcript_id')
        if tid is None:
            continue
        if feature == 'transcript':
            tags = attrs.get('tag', [])
            if isinstance(tags, str):
                tags = [tags]
            transcripts[tid] = {
                'gene_name'  : gene_name,
                'tags'       : set(tags),
                'has_cds'    : False,
                'gtf_length' : abs(int(parts[4]) - int(parts[3])) + 1,  # genomic span
            }
        elif feature == 'exon':
            length = abs(int(parts[4]) - int(parts[3])) + 1
            exon_lengths[tid] += length
        elif feature == 'CDS':
            if tid in transcripts:
                transcripts[tid]['has_cds'] = True

# Attach exonic length
for tid, info in transcripts.items():
    info['exonic_length'] = exon_lengths.get(tid, 0)

print(f'Parsed in {time.time()-t0:.0f}s')
print(f'Total transcripts collected: {len(transcripts):,}')

# Group by gene
by_gene = defaultdict(list)
for tid, info in transcripts.items():
    by_gene[info['gene_name']].append((tid, info))
print(f'Genes with at least 1 transcript: {len(by_gene)} / {len(all_genes)}')
missing = [g for g in all_genes if g not in by_gene]
if missing:
    print(f'MISSING (no transcript in GENCODE v44): {missing}')

In [ ]:
# Select one canonical transcript per gene using priority rules:
#   1. MANE_Select tag (clinical standard, single transcript per gene)
#   2. Ensembl_canonical tag
#   3. Longest CDS-containing transcript
#   4. Longest transcript overall

def pick_canonical(transcripts_list):
    """Pick one canonical transcript using priority hierarchy."""
    # Priority 1: MANE_Select
    mane = [t for t in transcripts_list if 'MANE_Select' in t[1]['tags']]
    if mane:
        return mane[0]
    # Priority 2: Ensembl_canonical
    can = [t for t in transcripts_list if 'Ensembl_canonical' in t[1]['tags']]
    if can:
        return can[0]
    # Priority 3: longest CDS-containing
    coding = [t for t in transcripts_list if t[1]['has_cds']]
    if coding:
        return max(coding, key=lambda t: t[1]['exonic_length'])
    # Priority 4: longest overall
    return max(transcripts_list, key=lambda t: t[1]['exonic_length'])

gene_length = {}
selection_method = {}
for gene in all_genes:
    if gene not in by_gene:
        gene_length[gene] = None
        continue
    tid, info = pick_canonical(by_gene[gene])
    gene_length[gene] = info['exonic_length']
    if 'MANE_Select' in info['tags']:
        method = 'MANE_Select'
    elif 'Ensembl_canonical' in info['tags']:
        method = 'Ensembl_canonical'
    elif info['has_cds']:
        method = 'longest_CDS'
    else:
        method = 'longest_overall'
    selection_method[gene] = (method, tid)

# Summary of selection methods used
from collections import Counter
method_counts = Counter(m for m, _ in selection_method.values())
print('Selection method distribution:')
for method, count in method_counts.most_common():
    print(f'  {method}: {count}')

# Verify all genes have a length
missing_len = [g for g in all_genes if gene_length[g] is None]
if missing_len:
    print(f'\nWARNING: {len(missing_len)} genes have no length: {missing_len}')
else:
    print('\n✓ All 88 genes have canonical transcript length assigned')

In [ ]:
# Sanity check on key gene lengths (known values from Ensembl/RefSeq)
# These are rough expected sizes (canonical MANE Select transcript)
expected_ranges = {
    'HBB'  : (500, 800),       # short β-globin transcript ~626 bp
    'HBA1' : (500, 700),       # α-globin ~577 bp
    'GATA1': (1500, 2500),     # GATA1 transcript ~2 kb
    'RUNX1': (5000, 9000),     # RUNX1 has long 3'UTR
    'CD19' : (2000, 4000),     # CD19 ~2.5 kb
    'PAX5' : (4000, 7000),
    'CD3D' : (700, 1500),      # short T-cell co-receptor
    'CD4'  : (2500, 4500),
    'CD8A' : (1500, 3000),
    'MPO'  : (2500, 4500),
    'SPI1' : (3500, 6000),
    'NKG7' : (300, 800),       # very short transcript
}
print('Sanity check on transcript lengths (rough expected ranges):')
for gene, (lo, hi) in expected_ranges.items():
    if gene in gene_length and gene_length[gene] is not None:
        L = gene_length[gene]
        method, tid = selection_method[gene]
        flag = '✓' if lo <= L <= hi else ('?' if L < lo*0.5 or L > hi*2 else '~')
        print(f'  [{flag}] {gene:6s}  L={L:>5}  (expected ~{lo}-{hi})  via {method:18s}  {tid}')
    else:
        print(f'  [✗] {gene}: NO LENGTH')

In [ ]:
# Save in yeast format: SINGLE LINE, tab-separated, NO trailing tab
# Yeast code:
#   for each in Column_order:
#     if (1+Column_order.index(each)) == len(Column_order):
#         outfile.write(ssTFs_len[each])         # last item, no trailing tab
#     else:
#         outfile.write(ssTFs_len[each]+'\t')

lengths_in_order = []
for gene in all_genes:
    L = gene_length[gene]
    if L is None:
        raise ValueError(f'Missing length for {gene} — cannot write file with gaps')
    lengths_in_order.append(str(L))

out_text = '\t'.join(lengths_in_order)   # tab between, no trailing tab, no newline

out_path = OUTPUT_DIR / 'BMMC_gene_length.txt'
with open(out_path, 'w') as f:
    f.write(out_text)
print(f'Saved: {out_path}  ({len(out_text):,} chars)')
print(f'First 200 chars:')
print(out_text[:200])
print(f'\nLast 100 chars:')
print(out_text[-100:])

# Save companion lookup TSV for reference (gene, length, source transcript)
lookup_path = OUTPUT_DIR / 'BMMC_gene_length_lookup.tsv'
with open(lookup_path, 'w') as f:
    f.write('order_idx\tgene_symbol\tlength_bp\tselection_method\ttranscript_id\n')
    for i, gene in enumerate(all_genes):
        method, tid = selection_method[gene]
        f.write(f'{i}\t{gene}\t{gene_length[gene]}\t{method}\t{tid}\n')
print(f'Saved: {lookup_path}')

# Final verification: parse it back and check
with open(out_path) as f:
    text_back = f.read()
parsed = text_back.split('\t')
print(f'\nVerification: read-back gives {len(parsed)} tokens (expected {len(all_genes)} = {len(all_genes)})')
assert len(parsed) == len(all_genes), 'Token count mismatch!'
assert all(p.isdigit() for p in parsed), 'Non-numeric tokens found!'
print(f'✓ All {len(parsed)} tokens are integers')

In [ ]:
# Methods summary
print('=' * 60)
print('Methods summary:')
print('=' * 60)
print(f"""
Gene lengths for the 88 genes in the BMMC panel were retrieved from the
GENCODE v44 (Ensembl primary annotation, release matching the ChIP prior
TSS source). For each gene, a canonical transcript was selected using the
following priority hierarchy: (1) MANE_Select tagged transcript, (2)
Ensembl_canonical, (3) longest CDS-containing transcript, (4) longest
transcript by total exonic length. The exonic length (sum of all exon
spans) of the selected canonical transcript was used as the gene length.
Gene lengths are stored in tab-separated single-line format, in the same
gene order as the expression matrix and binding priors.
""")